In [4]:
import joblib
import pandas as pd
import numpy as np
import seaborn as sb
%matplotlib inline
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, Normalizer, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import RFE  
#from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
from sklearn.linear_model import LogisticRegression

In [5]:
dataset=pd.read_csv('RecommendationData.csv')
df=dataset.copy()

In [6]:
df.head(5)

,destination,passenger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,...,CoffeeHouse,CarryAway,RestaurantLessThan20,Restaurant20To50,toCoupon_GEQ5min,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,Y
0,No Urgent Place,Alone,Sunny,55,2PM,Restaurant(<20),1d,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,0,0,0,1,1
1,No Urgent Place,Friend(s),Sunny,80,10AM,Coffee House,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,0,0,0,1,0
2,No Urgent Place,Friend(s),Sunny,80,10AM,Carry out & Take away,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,1
3,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,2h,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,0
4,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,1d,Female,21,Unmarried partner,...,never,NaN,4~8,1~3,1,1,0,0,1,0


In [7]:
df.columns

Index(['destination', 'passenger', 'weather', 'temperature', 'time', 'coupon',
       'expiration', 'gender', 'age', 'maritalStatus', 'has_children',
       'education', 'occupation', 'income', 'car', 'Bar', 'CoffeeHouse',
       'CarryAway', 'RestaurantLessThan20', 'Restaurant20To50',
       'toCoupon_GEQ5min', 'toCoupon_GEQ15min', 'toCoupon_GEQ25min',
       'direction_same', 'direction_opp', 'Y'],
      dtype='object')

In [8]:
df.shape

(12684, 26)

In [9]:
df.Y.value_counts()

Y
1    7210
0    5474
Name: count, dtype: int64

In [10]:
#drop duplicates features

df.drop_duplicates(inplace=True)
df=df.drop(columns=['car', 'toCoupon_GEQ5min'])   #we drop car because it has very high percentage pf missing values


In [11]:
# we have a small dataset, so we fill for missing values instead of dropping

from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy='most_frequent')
columns=['Bar', 'CoffeeHouse','CarryAway', 'RestaurantLessThan20','Restaurant20To50']
df[columns] = imputer.fit_transform(df[columns])

In [12]:
for i in df.columns:
    print ('\ncolumn name:', i, '\nunique values: ', df[i].unique())
    print('-----------------------------------')


column name: destination 
unique values:  ['No Urgent Place' 'Home' 'Work']
-----------------------------------

column name: passenger 
unique values:  ['Alone' 'Friend(s)' 'Kid(s)' 'Partner']
-----------------------------------

column name: weather 
unique values:  ['Sunny' 'Rainy' 'Snowy']
-----------------------------------

column name: temperature 
unique values:  [55 80 30]
-----------------------------------

column name: time 
unique values:  ['2PM' '10AM' '6PM' '7AM' '10PM']
-----------------------------------

column name: coupon 
unique values:  ['Restaurant(<20)' 'Coffee House' 'Carry out & Take away' 'Bar'
 'Restaurant(20-50)']
-----------------------------------

column name: expiration 
unique values:  ['1d' '2h']
-----------------------------------

column name: gender 
unique values:  ['Female' 'Male']
-----------------------------------

column name: age 
unique values:  ['21' '46' '26' '31' '41' '50plus' '36' 'below21']
-----------------------------------

column 

In [13]:
#formating the income column and and create an income midpoint


cleaned = df['income'].str.replace(r'[$,]', "", regex=True) # Step 1: Clean $ and commas
cleaned = cleaned.str.replace(" or More", "", regex=False) # Step 2: Handle "or More"
cleaned = cleaned.str.replace("Less than ", "0-", regex=False) # Step 3: Handle "Less than ..." explicitly
df[['income_min', 'income_max']] = cleaned.str.split('-', expand=True) # Step 4: Split into min and max
df['income_min'] = pd.to_numeric(df['income_min'], errors='coerce')# Step 5: Convert to numeric safely
df['income_max'] = pd.to_numeric(df['income_max'], errors='coerce')
df['income_max'] = df['income_max'].fillna(df['income_min']) # Step 6: Fill missing max values (like "100000 or More")
df['income_midpoint'] = df[['income_min', 'income_max']].mean(axis=1) # Step 7: Midpoint

df = df.drop(columns=['income']) # Step 8: Drop original column

In [14]:
#foromating the time column

df['time_24hr'] = pd.to_datetime(df['time'], format='%I%p', errors='coerce').dt.strftime('%H:%M')

df['time']=pd.to_datetime(df['time'], format='%I%p').dt.hour

df['time_24hr'] = pd.to_datetime(df['time'], format='%I%p', errors='coerce').dt.strftime('%H:%M')

df['hour']=pd.to_datetime(df['time'], format='%I%p', errors='coerce').dt.hour

def period_of_day(hour):
    if hour<12:
        return 'Morning'
    elif hour < 18:
        return "Afternoon"
    else:
        return 'Evening'
df['period']=df['hour'].apply(period_of_day)
df=df.drop(columns=['time_24hr','hour'])

In [15]:
#formating th expiration column

def parse_expiration(value):
    if value.endswith('d'): #day to hours
        return int(value[:-1])*24
    elif value.endswith ('h'): #HOURS
        return int(value[:-1])
    elif value.endswith ('m'): #minutes to fractio of hours
        return int(value[:-1])/60
    else:
        return None
df['expiration']=df['expiration'].apply(parse_expiration)
        

In [16]:
#formating and mapping Bar', 'CoffeeHouse', 'CarryAway', 'RestaurantLessThan20', 'Restaurant20To50

def clean_freq_col(df, column_name):
    mapping = {
        'never': 0,
        'less1': 0.5,
        '1~3': 2,
        '4~8': 6,
        'gt8': 9
    }
    df[column_name] = (
        df[column_name]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
    )
    return df

# Apply to each column
cols = ['Bar', 'CoffeeHouse', 'CarryAway', 'RestaurantLessThan20', 'Restaurant20To50']
for c in cols:
    df = clean_freq_col(df, c)


In [17]:
#encoding and formating education

# First strip whitespace
df['education'] = (df['education'].astype(str).str.strip())

# Then apply value replacement using a mapping dictionary
mapping = {
    'Some college - no degree': 'no degree',
    'Some High School': 'High School dropout'
}
df['education'] = df['education'].replace(mapping)

#mapping and assigning order
def education_parsed(df, column):
    mapping={
        'no degree':0,
        'High School dropout':1,
        'High School Graduate':2,
        'Bachelors degree':3,
        'Graduate degree (Masters or Doctorate)':4,
        'Associates degree':5
    }

    df[column]=(df[column].map(mapping))
    return df

df=education_parsed(df, 'education')

In [18]:
# we apply rare category grouping for features with low value counts
#Farming Fishing & Forestry      43
#Building & Grounds Cleaning     44
#Production Occupations          87

threshold=100
freq=df['occupation'].value_counts()
rare=freq[freq<threshold].index
df['occupation']=df['occupation'].replace(rare, 'others')

In [19]:
#format and encode age 

def age_encoder(df, column):
    mapping={
        '21':21,
        '46':46,
        '26':26,
        '31':31,
        '41':41,
        '50plus':55,
        '36':36,
        'below21':18
    }
    df['age']=df['age'].map(mapping)
    return df

df=age_encoder(df, 'age')

In [20]:
#constructing relationnship in our features

df['coffee_bar'] = (df['CoffeeHouse'] * df['Bar']) #social lifestyle interaction
df['restaurant_total'] = (df['RestaurantLessThan20'] * df['Restaurant20To50']) #restaurant frequency
df['CoffeeHouse_age']=df['CoffeeHouse']+df['age']

df['weather_destination'] = (df['weather'].astype(str) + '_' + df['destination'].astype(str)) 
df['occupation_coupon']=(df['occupation'].astype(str) + '_' + df['coupon'].astype(str)) 

In [21]:
smote=SMOTE(random_state=42, k_neighbors=5)
seed= 30

#split data for training

y=df['Y']
x=df.drop(columns=['Y'])

numCol=x.select_dtypes(include=[np.number]).columns
catcol=x.select_dtypes(include='object').columns

#we have to passthrough the binary feature [0,1 ie yes or no features]. we do not scale such features

binary_columns=[
    col for col in x.select_dtypes(include=[np.number]).columns
    if set(x[col].dropna().unique()).issubset({0,1})              #these are binary values
]
non_binary_columns=[col for col in numCol if col not in binary_columns ] #these are continuous values

#preprocessors
preprocessor=ColumnTransformer(transformers=[
    #('scaler', StandardScaler(), non_binary_columns),  no need scaling for random forest or tree based algorithms
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), catcol),
    ('non_binary_columns', 'passthrough', non_binary_columns),
    ('binary_columns', 'passthrough', binary_columns)
],remainder='drop')


### training

In [22]:
#===============
#models
#==============

model={
    'LogisticRegression':LogisticRegression(max_iter=2000, class_weight='balanced'),
    'RandomForest':RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed),
    'extraTrees':ExtraTreesClassifier(n_estimators=300, class_weight='balanced',random_state=seed),
    'GradientBoosting':GradientBoostingClassifier(),
    'XGBoost':XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=seed, eval_metric='logloss'),
    'LightGBM':LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight='balanced', random_state=seed) 
}

#===================
#CROSS VALIDATION
#====================
cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# ============================================
# EVALUATION LOOP
# ============================================
results=[]
for name, model in model.items():

    #create pipeline
    pipeline=Pipeline(steps=[
    ('preprocessor', preprocessor),
    #('smote', smote),
    ('model', model)])

    #cross validation
    scores=cross_val_score(pipeline,x,y, cv=cv, scoring='f1', n_jobs=-1)

    # CROSS-VALIDATED PREDICTIONS=
    y_pred=cross_val_predict(pipeline,x,y, cv=cv, n_jobs=-1)

    #CLASSISIFICATION REPORT
    print (f'\nclassificationreport :{name} \n')
    print(classification_report(y, y_pred))

    #CONFUSION MATRIX
    print('\nconfusin matrix\n')
    print(confusion_matrix(y, y_pred))

    #store results
    results.append({
        'model': name,
        'mean_f1': scores.mean(),
        'std_f1': scores.std()
    })

    print(f'{name}')
    print(f'mean f1: {scores.mean():.2f}')
    print (f'sdt f1: {scores.std():.2f}')
    print('=========================================')


classificationreport :LogisticRegression 

              precision    recall  f1-score   support

           0       0.57      0.63      0.60      5453
           1       0.69      0.64      0.66      7157

    accuracy                           0.63     12610
   macro avg       0.63      0.63      0.63     12610
weighted avg       0.64      0.63      0.63     12610


confusin matrix

[[3416 2037]
 [2585 4572]]
LogisticRegression
mean f1: 0.66
sdt f1: 0.01

classificationreport :RandomForest 

              precision    recall  f1-score   support

           0       0.74      0.66      0.70      5453
           1       0.76      0.83      0.79      7157

    accuracy                           0.76     12610
   macro avg       0.75      0.74      0.75     12610
weighted avg       0.75      0.76      0.75     12610


confusin matrix

[[3616 1837]
 [1244 5913]]
RandomForest
mean f1: 0.79
sdt f1: 0.01

classificationreport :extraTrees 

              precision    recall  f1-score   suppor

In [23]:
# ============================================
# RESULTS DATAFRAME
# ============================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by='mean_f1', ascending=False)
results_df

,model,mean_f1,std_f1
1,RandomForest,0.793269,0.010644
4,XGBoost,0.786685,0.009205
2,extraTrees,0.776531,0.009028
5,LightGBM,0.775436,0.011951
3,GradientBoosting,0.759828,0.013707
0,LogisticRegression,0.664210,0.011337


In [24]:
# ============================================
# FIT PIPELINE
# ============================================

pipeline.fit(x, y)

# ============================================
# ACCESS FITTED PREPROCESSOR
# ============================================

processor_fitted = pipeline.named_steps['preprocessor']

# ============================================
# TRANSFORM FEATURES
# ============================================

x_transformed = processor_fitted.transform(x)

# ============================================
# CONVERT SPARSE MATRIX TO DENSE
# ============================================

x_transformed = x_transformed.toarray()

# ============================================
# EXTRACT FEATURE NAMES
# ============================================

# onehot encoded columns
onehot_columns = processor_fitted.named_transformers_[

    'onehot'

].get_feature_names_out(catcol)

# continuous columns
continuous_columns = non_binary_columns

# binary columns
binary_cols = binary_columns

# combine all column names
all_columns = np.concatenate([

    onehot_columns,
    continuous_columns,
    binary_cols

])

# ============================================
# CREATE TRANSFORMED DATAFRAME
# ============================================

x_transformed_df = pd.DataFrame(

    x_transformed,
    columns=all_columns,
    index=x.index

)

# ============================================
# PREDICTIONS
# ============================================

y_pred = pipeline.predict(x)

# prediction probabilities
y_prob = pipeline.predict_proba(x)[:, 1]

# ============================================
# APPEND RESULTS
# ============================================

x_transformed_df['actual'] = y.values

x_transformed_df['prediction'] = y_pred

#x_transformed_df['probability'] = y_prob


[LightGBM] [Info] Number of positive: 7157, number of negative: 5453
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000728 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 467
[LightGBM] [Info] Number of data points in the train set: 12610, number of used features: 178
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
x_transformed_df.head(5)

,destination_No Urgent Place,destination_Work,passenger_Friend(s),passenger_Kid(s),passenger_Partner,weather_Snowy,weather_Sunny,coupon_Carry out & Take away,coupon_Coffee House,coupon_Restaurant(20-50),...,coffee_bar,restaurant_total,CoffeeHouse_age,has_children,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,actual,prediction
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,12.0,21.0,1.0,0.0,0.0,0.0,1.0,1,1
1,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,12.0,21.0,1.0,0.0,0.0,0.0,1.0,0,0
2,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,12.0,21.0,1.0,1.0,0.0,0.0,1.0,1,1
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,12.0,21.0,1.0,1.0,0.0,0.0,1.0,0,0
4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,12.0,21.0,1.0,1.0,0.0,0.0,1.0,0,0


In [26]:
joblib.dump(pipeline, 'coupon_recommender.pkl')

['coupon_recommender.pkl']